# DM Analysis Notebook (Spark)
This notebook analyses sample election results with explanatory markdown between each code cell, using PySpark for parallel file loading and computation.

In [2]:
from pathlib import Path  # Work with file-system paths safely.
from pyspark.sql import SparkSession  # Create and configure a Spark session.
from pyspark.sql import functions as F  # Use Spark SQL functions for transformations.
from pyspark.sql.window import Window  # Define window functions for winner selection.

spark_builder = SparkSession.Builder()

spark = (
    spark_builder
    .appName("DM Analysis Spark")
    .master("spark://localhost:7077")  # Connect to Docker Spark master.
    .config("spark.executor.instances", "3")  # Target three worker executors.
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")  # Keep notebook output focused on results.
print(f"Spark master: {spark.sparkContext.master}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 23:00:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark master: spark://localhost:7077


## Step 1: Locate Input Files
The next code cell discovers all sample result JSON files.

In [3]:
base_path = Path("resources/sample-election-results")  # Point to the local sample results directory.
spark_base_path = "file:///tmp/election-data"  # Shared absolute path copied to host and Spark workers.
json_files = sorted(base_path.glob("*.json"))  # Collect and sort every JSON result file.
print(f"Files found: {len(json_files)}")  # Show how many files were discovered.
print(json_files[:5])  # Preview the first few paths for validation.

Files found: 650
[PosixPath('resources/sample-election-results/result001.json'), PosixPath('resources/sample-election-results/result002.json'), PosixPath('resources/sample-election-results/result003.json'), PosixPath('resources/sample-election-results/result004.json'), PosixPath('resources/sample-election-results/result005.json')]


## Step 2: Load and Validate Records in Parallel
The next cell loads all files at once through Spark's distributed JSON reader so parsing runs in parallel.

In [4]:
results_df = (
    spark.read
    .option("multiLine", True)
    .json(f"{spark_base_path}/*.json")  # Read all JSON files in parallel from worker-visible path.
)

results_df = results_df.cache()  # Cache because the DataFrame is reused in later steps.
print(f"Records loaded: {results_df.count()}")  # Trigger read and confirm row count.
results_df.printSchema()  # Preview the inferred schema.

Records loaded: 650
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- partyResults: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- party: string (nullable = true)
 |    |    |-- share: double (nullable = true)
 |    |    |-- votes: long (nullable = true)
 |-- seqNo: long (nullable = true)



## Step 3: Compute Basic Metrics
This cell computes constituency counts and total-vote statistics from party results using Spark aggregations.

In [5]:
# Build a resilient constituency identifier by taking the first non-null value among id, name, or file path.
constituency_key = F.coalesce(  # F.coalesce returns the first non-null expression in order.
    F.col("id").cast("string"),  # F.col reads column id and cast ensures consistent string type.
    F.col("name").cast("string"),  # F.col on name provides a fallback key if id is missing.
    F.input_file_name()  # F.input_file_name adds the source filename as the final fallback key.
 )

display(results_df.select(constituency_key.alias("constituency_key")).show(20, False))  # Show the first 20 constituency keys to verify the coalescing logic.

# Create one total-votes row per constituency from nested partyResults entries.
constituency_totals = (
    results_df  # Start from the raw election results DataFrame.
    .select(constituency_key.alias("constituency_key"), F.explode_outer("partyResults").alias("party_result"))  # select keeps needed fields and explode_outer expands partyResults safely even when null.
    .groupBy("constituency_key")  # groupBy groups rows so each constituency is aggregated independently.
    .agg(F.sum(F.coalesce(F.col("party_result.votes"), F.lit(0))).alias("total_votes"))  # agg with sum computes constituency total votes, defaulting missing votes to 0 via F.lit.
)

display(constituency_totals.select("constituency_key", "total_votes").orderBy(F.desc("total_votes")).limit(10).show(20, False))  # Show top 10 constituencies by total votes for validation.

# Aggregate dataset-level summary metrics from per-constituency totals.
metrics = constituency_totals.agg(
    F.count("*").alias("total_constituencies"),  # Count how many constituency groups exist.
    F.avg("total_votes").alias("average_total_votes"),  # Compute mean total votes per constituency.
    F.max("total_votes").alias("max_total_votes"),  # Capture the largest constituency vote total.
    F.min("total_votes").alias("min_total_votes")  # Capture the smallest constituency vote total.
).collect()[0]  # collect brings the single aggregated Row to the driver and [0] extracts it.

display(metrics.asDict())  # Show the computed metrics as a dictionary for clarity.
 
# Print a compact, human-readable summary of the computed metrics.
print(f"Constituencies: {metrics['total_constituencies']}")
print(f"Average total votes: {metrics['average_total_votes']:.2f}")
print(f"Max total votes: {metrics['max_total_votes']:.2f}")
print(f"Min total votes: {metrics['min_total_votes']:.2f}")

+----------------+
|constituency_key|
+----------------+
|499             |
|262             |
|104             |
|393             |
|220             |
|374             |
|345             |
|213             |
|533             |
|558             |
|446             |
|436             |
|23              |
|348             |
|479             |
|441             |
|224             |
|228             |
|45              |
|71              |
+----------------+
only showing top 20 rows



None

+----------------+-----------+
|constituency_key|total_votes|
+----------------+-----------+
|333             |66843      |
|564             |56649      |
|546             |53775      |
|412             |53764      |
|26              |53685      |
|519             |53350      |
|200             |53225      |
|235             |53187      |
|554             |53147      |
|121             |52990      |
+----------------+-----------+



None

{'total_constituencies': 650,
 'average_total_votes': 41803.11692307692,
 'max_total_votes': 66843,
 'min_total_votes': 14884}

Constituencies: 650
Average total votes: 41803.12
Max total votes: 66843.00
Min total votes: 14884.00


## Step 4: Quick Party Frequency Scan
The final code cell derives each winner from party vote totals and counts party wins using window functions.

In [6]:
# Part 1: Flatten party vote rows per constituency
party_votes = (
    results_df
    .select(constituency_key.alias("constituency_key"), F.explode_outer("partyResults").alias("party_result"))
    .select(
        "constituency_key",
        F.coalesce(F.col("party_result.party"), F.lit("UNKNOWN")).alias("party"),
        F.coalesce(F.col("party_result.votes"), F.lit(0)).alias("votes")
    )
)
print("Part 1 complete: party_votes")
print(f"Rows: {party_votes.count()}")
display(party_votes.orderBy(F.col("votes").desc()).limit(10))

# Part 2: Pick one winner per constituency (highest votes, party name tie-break)
winner_window = Window.partitionBy("constituency_key").orderBy(F.col("votes").desc(), F.col("party").asc())
winners = (
    party_votes
    .withColumn("rank", F.row_number().over(winner_window))
    .filter(F.col("rank") == 1)
    .select("constituency_key", "party", "votes")
)
print("Part 2 complete: winners")
print(f"Constituencies with winners: {winners.count()}")
display(winners.limit(10))

# Part 3: Count winner frequency by party
party_counts = (
    winners
    .groupBy("party")
    .count()
    .orderBy(F.col("count").desc(), F.col("party").asc())
)
print("Part 3 complete: party_counts (top 10)")
display(party_counts.limit(10))

# Optional text output
for row in party_counts.limit(10).collect():
    print(f"{row['party']}: {row['count']}")

Part 1 complete: party_votes
Rows: 3626


DataFrame[constituency_key: string, party: string, votes: bigint]

Part 2 complete: winners
Constituencies with winners: 650


DataFrame[constituency_key: string, party: string, votes: bigint]

Part 3 complete: party_counts (top 10)


DataFrame[party: string, count: bigint]

LAB: 349
CON: 210
LD: 62
DUP: 9
SNP: 6
SF: 5
SDLP: 3
PC: 2
IKHH: 1
OTH: 1


In [9]:
# Bonus scoreboard fields: total votes and vote share (%)
party_vote_totals = (
    party_votes
    .groupBy("party")
    .agg(F.sum("votes").alias("total_votes"))
)

overall_votes = party_vote_totals.agg(F.sum("total_votes").alias("overall_votes"))

scoreboard = (
    party_counts
    .join(party_vote_totals, on="party", how="left")
    .crossJoin(overall_votes)
    .withColumn(
        "vote_share_pct",
        F.when(F.col("overall_votes") > 0, (F.col("total_votes") / F.col("overall_votes")) * 100).otherwise(F.lit(0.0))
    )
    .select(
        "party",
        F.col("count").alias("constituencies_won"),
        "total_votes",
        F.round("vote_share_pct", 2).alias("vote_share_pct")
    )
    .orderBy(F.col("constituencies_won").desc(), F.col("total_votes").desc(), F.col("party").asc())
)

scoreboard.show()  # Show the top 20 parties in the scoreboard with all metrics.

+-----+------------------+-----------+--------------+
|party|constituencies_won|total_votes|vote_share_pct|
+-----+------------------+-----------+--------------+
|  LAB|               349|    9567584|         35.21|
|  CON|               210|    8784915|         32.33|
|   LD|                62|    5985454|         22.03|
|  DUP|                 9|     241856|          0.89|
|  SNP|                 6|     412267|          1.52|
|   SF|                 5|     174530|          0.64|
| SDLP|                 3|     125626|          0.46|
|   PC|                 2|     174838|          0.64|
|  OTH|                 1|     520736|          1.92|
|  UUP|                 1|     127314|          0.47|
|  RES|                 1|      30190|          0.11|
| IKHH|                 1|      18739|          0.07|
+-----+------------------+-----------+--------------+



## Optional Cleanup
Stop Spark when you are done to release local resources.

In [26]:
spark.stop()